Thin slice; end to end

Build a small end-to-end pipeline that incorporates every layer of the final system to identify any potential integration problems.

Scope:
- Division: women's strawweight
- Time: 2021 onwards
- Sample: 10 fighters with relevant Trends presence
- Competitive axis: opponent-weighted win rate (PLACEHOLDER, not Glicko-2)
- Public profile axis: single Trends 12-month average (PLACEHOLDER, not the full composite)

This is the early thin slice proof of concept, run before the full build to validate the end to end pipeline shape on a handful of fighters. It deliberately uses Google Trends and a placeholder competitive score, both later superseded: the competitive axis became Glicko-2, and the public profile axis became a Wikipedia and GDELT composite after this run surfaced the Trends normalisation problem recorded in the findings below. It is retained as the record of that validation and that finding, not as part of the final pipeline.



## Section 1: Setup and imports

In [1]:
# BLOCK 1: Imports and config

import pandas as pd
import numpy as np
import time
from datetime import datetime
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
!pip install pytrends
from pytrends.request import TrendReq

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Lock scope
DIVISION = 'Women\'s Strawweight'
PERIOD_START = '2021-01-01'
N_FIGHTERS_TO_PLOT = 10
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Time and date of last run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Division: {DIVISION}")
print(f"Timeframe: {PERIOD_START} onwards")

Time and date of last run: 2026-08-26 20:46:59
Division: Women's Strawweight
Timeframe: 2021-01-01 onwards


## Section 2: Load and filter UFCStats data

Pull the Greco1899 CSVs, filter to the scope and produce a clean per fight df.

In [2]:
# BLOCK 2: Pull Greco1899 CSVs

BASE_URL = "https://raw.githubusercontent.com/Greco1899/scrape_ufc_stats/main/"

files_needed = {
    'fight_results': 'ufc_fight_results.csv',
    'events': 'ufc_event_details.csv',
    'fighter_details': 'ufc_fighter_details.csv',
}

dfs = {}
for key, fname in files_needed.items():
    dfs[key] = pd.read_csv(BASE_URL + fname)

df_fights = dfs['fight_results'].copy()
df_events = dfs['events'].copy()
df_fighters = dfs['fighter_details'].copy()

print(f"Done")

Done


In [3]:
# BLOCK 3: Filter to thin slice scope

# EVENT/BOUT whitespace strip + join on EVENT for dates


# String hygiene
df_fights['EVENT'] = df_fights['EVENT'].str.strip()
df_fights['BOUT'] = df_fights['BOUT'].str.strip()
df_events['EVENT'] = df_events['EVENT'].str.strip()

# Parse dates and join
df_events['DATE_PARSED'] = pd.to_datetime(df_events['DATE'], errors='coerce')
df_fights = df_fights.merge(
    df_events[['EVENT', 'DATE_PARSED']],
    on='EVENT',
    how='left'
)

# Filter division
mask_division = df_fights['WEIGHTCLASS'].str.contains(DIVISION, na=False)
mask_period = df_fights['DATE_PARSED'] >= PERIOD_START
df_fights_scoped = df_fights[mask_division & mask_period].copy()

# Filter out bad outcomes
modelable_outcomes = ['W/L', 'L/W', 'D/D']
df_fights_scoped = df_fights_scoped[df_fights_scoped['OUTCOME'].isin(modelable_outcomes)]

print(f"Filtered to {DIVISION} from {PERIOD_START} onwards: {len(df_fights_scoped):,} fights")
print()
print("Sample:")
df_fights_scoped[['EVENT', 'BOUT', 'OUTCOME', 'METHOD', 'DATE_PARSED']].head()

Filtered to Women's Strawweight from 2021-01-01 onwards: 190 fights

Sample:


,EVENT,BOUT,OUTCOME,METHOD,DATE_PARSED
12,UFC Fight Night: Hernandez vs. Rodrigues,Shanelle Dyer vs. Elise Reed,W/L,KO/TKO,2026-08-22
14,UFC 330: Makhachev vs. Machado Garry,Mackenzie Dern vs. Gillian Robertson,W/L,Decision - Unanimous,2026-08-15
28,UFC Fight Night: Gamrot vs. Salkilld,Amanda Lemos vs. Alexia Thainara,L/W,Decision - Unanimous,2026-08-08
36,UFC Fight Night: Gamrot vs. Salkilld,Gigi Canuto vs. Carol Foro,L/W,Decision - Unanimous,2026-08-08
45,Noche UFC: Lopes vs. Silva,Tatiana Suarez vs. Amanda Lemos,W/L,Decision - Unanimous,2025-09-13


## Section 3: Build fighter frame

Adjust data from per fight (one row, two fighters) to per fighter (one row per fighter with aggregate stats).


In [4]:
# BLOCK 4: Parse BOUT into two fighter columns and build fighter frame

def parse_bout_to_fighters(df):
    rows = []
    for _, fight in df.iterrows():
        # Parse the BOUT string. Expected format: "Fighter A vs. Fighter B"
        bout_parts = fight['BOUT'].split(' vs. ')
        if len(bout_parts) != 2:
            continue
        fighter_a, fighter_b = bout_parts[0].strip(), bout_parts[1].strip()

        # Parse OUTCOME
        outcome_parts = fight['OUTCOME'].split('/')
        if len(outcome_parts) != 2:
            continue
        result_a, result_b = outcome_parts[0], outcome_parts[1]

        # Add two rows, one per fighter
        rows.append({
            'FIGHTER': fighter_a,
            'OPPONENT': fighter_b,
            'RESULT': result_a,
            'EVENT': fight['EVENT'],
            'DATE': fight['DATE_PARSED'],
            'METHOD': fight['METHOD'],
        })
        rows.append({
            'FIGHTER': fighter_b,
            'OPPONENT': fighter_a,
            'RESULT': result_b,
            'EVENT': fight['EVENT'],
            'DATE': fight['DATE_PARSED'],
            'METHOD': fight['METHOD'],
        })

    return pd.DataFrame(rows)

df_per_fighter = parse_bout_to_fighters(df_fights_scoped)
print(f"Fighter rows: {len(df_per_fighter):,}")
print(f"Unique fighters: {df_per_fighter['FIGHTER'].nunique()}")
print()
print("Sample:")
df_per_fighter.head()

Fighter rows: 380
Unique fighters: 81

Sample:


,FIGHTER,OPPONENT,RESULT,EVENT,DATE,METHOD
0,Shanelle Dyer,Elise Reed,W,UFC Fight Night: Hernandez vs. Rodrigues,2026-08-22,KO/TKO
1,Elise Reed,Shanelle Dyer,L,UFC Fight Night: Hernandez vs. Rodrigues,2026-08-22,KO/TKO
2,Mackenzie Dern,Gillian Robertson,W,UFC 330: Makhachev vs. Machado Garry,2026-08-15,Decision - Unanimous
3,Gillian Robertson,Mackenzie Dern,L,UFC 330: Makhachev vs. Machado Garry,2026-08-15,Decision - Unanimous
4,Amanda Lemos,Alexia Thainara,L,UFC Fight Night: Gamrot vs. Salkilld,2026-08-08,Decision - Unanimous


## Section 4: Placeholder competitive axis

Compute an opponent weighted win rate as a fighter score; acts as a placeholder for the Glicko-2 rating.

In [5]:
# BLOCK 5: Placeholder competitive score (weighted win rate)

# Math:
# For each fighter, compute a basic win rate
# For each fight, look up the opponent's basic win rate
# Re-score: each win contributes opponent's win rate; each loss contributes (1 - opponent's win rate)
# Average across each fighter's fights

# basic win rate per fighter
df_per_fighter['IS_WIN'] = (df_per_fighter['RESULT'] == 'W').astype(int)
df_per_fighter['IS_LOSS'] = (df_per_fighter['RESULT'] == 'L').astype(int)
df_per_fighter['IS_DRAW'] = (df_per_fighter['RESULT'] == 'D').astype(int)

basic_winrate = df_per_fighter.groupby('FIGHTER').agg(
    wins=('IS_WIN', 'sum'),
    losses=('IS_LOSS', 'sum'),
    draws=('IS_DRAW', 'sum'),
    n_fights=('FIGHTER', 'count')
)
basic_winrate['basic_wr'] = basic_winrate['wins'] / basic_winrate['n_fights']

# weighted scoring per fight
df_per_fighter['opp_basic_wr'] = df_per_fighter['OPPONENT'].map(basic_winrate['basic_wr'])
df_per_fighter['weighted_score'] = np.where(
    df_per_fighter['RESULT'] == 'W', df_per_fighter['opp_basic_wr'],
    np.where(
        df_per_fighter['RESULT'] == 'L', 1 - df_per_fighter['opp_basic_wr'],
        0.5  # draw
    )
)

# average per fighter
competitive_scores = df_per_fighter.groupby('FIGHTER').agg(
    competitive_score=('weighted_score', 'mean'),
    n_fights=('FIGHTER', 'count')
).reset_index()

# Filter to fighters with at least 3 fights
competitive_scores = competitive_scores[competitive_scores['n_fights'] >= 3]
competitive_scores = competitive_scores.sort_values('competitive_score', ascending=False)

print(f"Fighters with competitive scores: {len(competitive_scores)}")
print("\nRanking by placeholder competitive score:")
print(competitive_scores.head(10).to_string(index=False))

Fighters with competitive scores: 56

Ranking by placeholder competitive score:
        FIGHTER  competitive_score  n_fights
 Rose Namajunas           0.642857         3
Gloria de Paula           0.579167         4
   Ashley Yoder           0.568376         3
  Jessica Penne           0.517172         5
 Mackenzie Dern           0.513137        11
   Amanda Ribas           0.497835         5
 Tatiana Suarez           0.497711         4
   Fatima Kline           0.482809         4
    Yan Xiaonan           0.476603         7
   Denise Gomes           0.473905         8


## Section 5: Select sample fighters and pull Trends data

Pick 10 fighters and pull their Google Trends data over the last 12 months.

In [6]:
# BLOCK 6: Select sample fighters

sample_fighters = competitive_scores.head(N_FIGHTERS_TO_PLOT)['FIGHTER'].tolist()
print(f"Sample fighters ({len(sample_fighters)}):")
for f in sample_fighters:
    print(f"  - {f}")

Sample fighters (10):
  - Rose Namajunas
  - Gloria de Paula
  - Ashley Yoder
  - Jessica Penne
  - Mackenzie Dern
  - Amanda Ribas
  - Tatiana Suarez
  - Fatima Kline
  - Yan Xiaonan
  - Denise Gomes


In [7]:
# BLOCK 7: Pull Trends data
#
# Uses pytrends, one fighter at a time with patient sleep between
# calls. Cache results to a CSV so reruns don't repeat the calls.

CACHE_PATH = OUTPUT_DIR / 'trends_cache.csv'

if CACHE_PATH.exists():
    print(f"Loading cached Trends data from {CACHE_PATH}")
    trends_data = pd.read_csv(CACHE_PATH)
else:
    print("No cache found; pulling fresh Trends data")
    pytrends = TrendReq(hl='en-US', tz=300, timeout=(10, 25))

    trends_rows = []
    failures = []

    for i, fighter in enumerate(sample_fighters):
        print(f"  [{i+1}/{len(sample_fighters)}] {fighter}")
        try:
            pytrends.build_payload(
                kw_list=[fighter],
                timeframe='today 12-m',
                geo='US'
            )
            df = pytrends.interest_over_time()
            if len(df) > 0 and fighter in df.columns:
                avg_interest = df[fighter].mean()
                trends_rows.append({
                    'FIGHTER': fighter,
                    'avg_interest_12m': avg_interest,
                    'max_interest_12m': df[fighter].max(),
                })
                print(f"    avg interest: {avg_interest:.2f}")
            else:
                print(f"    no data returned")
                failures.append(fighter)
        except Exception as e:
            print(f"    FAIL: {type(e).__name__}: {e}")
            failures.append(fighter)

        # Patient sleep to avoid rate limits
        time.sleep(15)

    trends_data = pd.DataFrame(trends_rows)
    trends_data.to_csv(CACHE_PATH, index=False)

    print(f"\nDone. Successes: {len(trends_rows)}, Failures: {len(failures)}")
    if failures:
        print(f"Failed fighters: {failures}")

print("\nTrends data:")
print(trends_data.to_string(index=False))

No cache found; pulling fresh Trends data
  [1/10] Rose Namajunas


/usr/local/lib/python3.13/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


    avg interest: 4.92
  [2/10] Gloria de Paula
    no data returned
  [3/10] Ashley Yoder


/usr/local/lib/python3.13/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


    avg interest: 2.62
  [4/10] Jessica Penne


...

  [9/10] Yan Xiaonan
    avg interest: 5.36


/usr/local/lib/python3.13/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


  [10/10] Denise Gomes


/usr/local/lib/python3.13/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


    avg interest: 4.45

Done. Successes: 9, Failures: 1
Failed fighters: ['Gloria de Paula']

Trends data:
       FIGHTER  avg_interest_12m  max_interest_12m
Rose Namajunas          4.924528               100
  Ashley Yoder          2.622642               100
 Jessica Penne          3.358491               100
Mackenzie Dern          8.056604               100
  Amanda Ribas          8.339623               100
Tatiana Suarez          5.415094               100
  Fatima Kline          7.358491               100
   Yan Xiaonan          5.358491               100
  Denise Gomes          4.452830               100


## Section 6: Build the joined two-axis frame

Join competitive scores and Trends data into one frame ready for plotting. Fighter names need to match across sources


In [8]:
# BLOCK 8: Join competitive and public profile axes

df_plot = competitive_scores.merge(
    trends_data,
    on='FIGHTER',
    how='inner'  # only keep fighters present in both
)

print(f"Fighters in both axes: {len(df_plot)} of {N_FIGHTERS_TO_PLOT} target")
if len(df_plot) < N_FIGHTERS_TO_PLOT:
    missing = set(competitive_scores.head(N_FIGHTERS_TO_PLOT)['FIGHTER']) - set(df_plot['FIGHTER'])
    print(f"Fighters dropped (no Trends data): {missing}")

print("\nFinal plot frame:")
print(df_plot.to_string(index=False))

Fighters in both axes: 9 of 10 target
Fighters dropped (no Trends data): {'Gloria de Paula'}

Final plot frame:
       FIGHTER  competitive_score  n_fights  avg_interest_12m  max_interest_12m
Rose Namajunas           0.642857         3          4.924528               100
  Ashley Yoder           0.568376         3          2.622642               100
 Jessica Penne           0.517172         5          3.358491               100
Mackenzie Dern           0.513137        11          8.056604               100
  Amanda Ribas           0.497835         5          8.339623               100
Tatiana Suarez           0.497711         4          5.415094               100
  Fatima Kline           0.482809         4          7.358491               100
   Yan Xiaonan           0.476603         7          5.358491               100
  Denise Gomes           0.473905         8          4.452830               100


## Section 7: Build the Plotly scatter

Generate the scatter and writes it to a standalone HTML file.


In [9]:
# BLOCK 9: Build the scatter and write HTML

fig = px.scatter(
    df_plot,
    x='competitive_score',
    y='avg_interest_12m',
    text='FIGHTER',
    hover_data=['n_fights', 'max_interest_12m'],
    title=f'Testing: {DIVISION} scatter view (placeholder scoring)',
    labels={
        'competitive_score': 'Competitive score (placeholder, opponent weighted win rate)',
        'avg_interest_12m': 'Public profile (12-month avg Trends interest)',
    }
)

fig.update_traces(
    textposition='top center',
    marker=dict(size=12)
)

fig.update_layout(
    width=900,
    height=600,
    template='plotly_white'
)

# Write HTML
html_path = OUTPUT_DIR / 'index.html'
fig.write_html(
    html_path,
    include_plotlyjs='cdn',  # use CDN hosted to keep file small
    full_html=True
)

print(f"HTML written: {html_path}")
print(f"File size: {html_path.stat().st_size:,} bytes")

# notebook view
fig.show()

HTML written: outputs/index.html
File size: 9,344 bytes


## Section 8: Findings

End to end run completed without errors on 2026-05-22. Every layer (UFCStats load, scope filter, bout parsing, placeholder competitive axis, Trends pull, join, Plotly HTML export) executed and produced a standalone 9.4 KB `index.html`. The integration path is sound; the issues below are about data behaviour, not pipeline breakage.

Integration worked, with placeholders.

Trends normalisation issue. Every fighter returned a max score of 100 because each was pulled in a separate request. Google Trends scales results relative to the single query, so each fighter's busiest week becomes her own 100 regardless of how popular she actually is. This means the averages (3.38 to 7.32) are each measured on a different ruler and cannot be compared across fighters as-is. To fix this, fighters need to be queried together in batches with a shared reference term so all values sit on one common scale. The proposal anticipated this; the thin slice confirms it needs solving before the public profile axis is built.

pytrends feasibility looks good at this scale. Ten sequential pulls with a 15-second sleep returned ten successes and zero failures, geo-restricted to US. This supports the patient-sequential approach for the real run, though ten fighters is a small test; rate-limit behaviour over hundreds of fighters still needs checking. Caching to CSV on first pull works, so reruns will not repeat the calls.

The two axes visibly diverge, which is the point! Gloria de Paula ranks third on the placeholder competitive score (4 fights) yet has the lowest Trends interest in the sample (3.38); Yan Xiaonan has the highest interest (7.32) but sits near the bottom on competitive score. Even with placeholder maths, the slice produces the two axis spread the framework is built around.

The draw-handling branch is untested. `D/D` is included in the modelable outcomes and the scoring code assigns draws 0.5, but the strawweight sample contained no draws.

Data is current. The slice includes a fight dated 2026-05-16, days before the run. The Greco1899 daily refresh is clearly working, but it also means counts and rankings will shift run to run; anything quoted from this slice is a snapshot, not a fixed figure.
